# Notebook 7 — Power BI Export

**What this notebook does:** Consolidates all the analysis outputs from
notebooks 1-6 into 7 clean, Power BI-ready CSV files.

**Input:** `data/processed/cleaned_churn.csv`, `outputs/model_comparison.csv`,
`outputs/shap_feature_importance.csv`, `data/powerbi/customer_risk_scores.csv`,
`data/powerbi/ab_test_results.csv`

**Output:** 7 CSV files in `data/powerbi/`

**Estimated run time:** ~10 seconds

In [1]:
# Imports and shared project utilities
import os
import sys
import pandas as pd
import numpy as np

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))
from utils import save_dataframe, load_csv_safely, project_path

powerbi_dir = project_path("data", "powerbi")
file_summary = []

## 1. Customer risk scores (already created in notebook 5)

In [2]:
required_notebook_5 = "05_risk_scoring.ipynb"
risk_scores_path = project_path("data", "powerbi", "customer_risk_scores.csv")
customer_risk_scores = load_csv_safely(risk_scores_path, required_notebook_5)

print(f"customer_risk_scores.csv: {customer_risk_scores.shape[0]} rows")
print(f"Columns: {list(customer_risk_scores.columns)}")
file_summary.append(("customer_risk_scores.csv", customer_risk_scores.shape[0], customer_risk_scores.shape[1], os.path.getsize(risk_scores_path)))

customer_risk_scores.csv: 7032 rows
Columns: ['customerID', 'gender', 'SeniorCitizen', 'Partner', 'tenure', 'Contract', 'MonthlyCharges', 'TotalCharges', 'InternetService', 'TechSupport', 'Churn', 'churn_probability', 'risk_segment']


## 2. Model comparison

In [3]:
required_notebook_3 = "03_model_training.ipynb"
model_comparison_source = load_csv_safely(project_path("outputs", "model_comparison.csv"), required_notebook_3)

model_comparison_export = model_comparison_source[["Model", "Accuracy", "Precision", "Recall", "F1", "AUC"]]
model_comparison_path = project_path("data", "powerbi", "model_comparison.csv")
save_dataframe(model_comparison_export, model_comparison_path)
file_summary.append(("model_comparison.csv", model_comparison_export.shape[0], model_comparison_export.shape[1], os.path.getsize(model_comparison_path)))

✅ Saved: /Users/prasadlola/Documents/customer_churn_project/data/powerbi/model_comparison.csv (478.0B) — 4 rows, 6 columns


## 3. Churn by segment

In [4]:
required_notebook_1 = "01_data_cleaning_eda.ipynb"
cleaned_data = load_csv_safely(project_path("data", "processed", "cleaned_churn.csv"), required_notebook_1)

# tenure_group isn't stored in cleaned_churn.csv (it's an engineered feature), so rebuild it here
tenure_bins = [0, 12, 24, 36, 48, 60, 72]
tenure_labels = ["0-12 months", "13-24 months", "25-36 months",
                 "37-48 months", "49-60 months", "61-72 months"]
cleaned_data["tenure_group"] = pd.cut(cleaned_data["tenure"], bins=tenure_bins, labels=tenure_labels, include_lowest=True)

segment_columns = ["Contract", "InternetService", "PaymentMethod", "SeniorCitizen", "tenure_group"]

segment_rows = []
for segment_type in segment_columns:
    grouped = cleaned_data.groupby(segment_type, observed=True)["Churn"].agg(["count", "sum"])
    for segment_value, row in grouped.iterrows():
        segment_rows.append({
            "segment_type": segment_type,
            "segment_value": str(segment_value),
            "total_customers": int(row["count"]),
            "churned_customers": int(row["sum"]),
            "churn_rate": round(row["sum"] / row["count"], 4),
        })

churn_by_segment = pd.DataFrame(segment_rows)
churn_by_segment_path = project_path("data", "powerbi", "churn_by_segment.csv")
save_dataframe(churn_by_segment, churn_by_segment_path)
file_summary.append(("churn_by_segment.csv", churn_by_segment.shape[0], churn_by_segment.shape[1], os.path.getsize(churn_by_segment_path)))

✅ Saved: /Users/prasadlola/Documents/customer_churn_project/data/powerbi/churn_by_segment.csv (810.0B) — 18 rows, 5 columns


## 4. A/B test results (already created in notebook 6)

In [5]:
required_notebook_6 = "06_ab_test.ipynb"
ab_test_path = project_path("data", "powerbi", "ab_test_results.csv")
ab_test_results = load_csv_safely(ab_test_path, required_notebook_6)

print(f"ab_test_results.csv: {ab_test_results.shape[0]} rows")
print(f"Columns: {list(ab_test_results.columns)}")
file_summary.append(("ab_test_results.csv", ab_test_results.shape[0], ab_test_results.shape[1], os.path.getsize(ab_test_path)))

ab_test_results.csv: 2 rows
Columns: ['group', 'total_customers', 'churned', 'churn_rate', 'lower_ci', 'upper_ci']


## 5. Feature importance (top 15 from notebook 4's SHAP values)

In [6]:
required_notebook_4 = "04_shap_explainability.ipynb"
shap_importance_source = load_csv_safely(project_path("outputs", "shap_feature_importance.csv"), required_notebook_4)

top_15_features = shap_importance_source.head(15).reset_index(drop=True)
feature_importance_export = pd.DataFrame({
    "rank": range(1, len(top_15_features) + 1),
    "feature": top_15_features["feature"],
    "mean_shap_value": top_15_features["mean_abs_shap_value"],
})

feature_importance_path = project_path("data", "powerbi", "feature_importance.csv")
save_dataframe(feature_importance_export, feature_importance_path)
file_summary.append(("feature_importance.csv", feature_importance_export.shape[0], feature_importance_export.shape[1], os.path.getsize(feature_importance_path)))

✅ Saved: /Users/prasadlola/Documents/customer_churn_project/data/powerbi/feature_importance.csv (583.0B) — 15 rows, 3 columns


## 6. Monthly revenue at risk (from customer risk scores)

In [7]:
revenue_at_risk_rows = []
for segment, segment_data in customer_risk_scores.groupby("risk_segment"):
    customer_count = len(segment_data)
    avg_monthly_charges = segment_data["MonthlyCharges"].mean()
    total_monthly_revenue = segment_data["MonthlyCharges"].sum()
    estimated_monthly_churn_revenue_loss = (segment_data["churn_probability"] * segment_data["MonthlyCharges"]).sum()

    revenue_at_risk_rows.append({
        "risk_segment": segment,
        "customer_count": customer_count,
        "avg_monthly_charges": round(avg_monthly_charges, 2),
        "total_monthly_revenue": round(total_monthly_revenue, 2),
        "estimated_monthly_churn_revenue_loss": round(estimated_monthly_churn_revenue_loss, 2),
    })

monthly_revenue_at_risk = pd.DataFrame(revenue_at_risk_rows).sort_values("risk_segment").reset_index(drop=True)
revenue_at_risk_path = project_path("data", "powerbi", "monthly_revenue_at_risk.csv")
save_dataframe(monthly_revenue_at_risk, revenue_at_risk_path)
file_summary.append(("monthly_revenue_at_risk.csv", monthly_revenue_at_risk.shape[0], monthly_revenue_at_risk.shape[1], os.path.getsize(revenue_at_risk_path)))

✅ Saved: /Users/prasadlola/Documents/customer_churn_project/data/powerbi/monthly_revenue_at_risk.csv (222.0B) — 3 rows, 5 columns


## 7. EDA summary

In [8]:
total_customers = len(cleaned_data)
churn_rate = cleaned_data["Churn"].mean()
avg_tenure = cleaned_data["tenure"].mean()
avg_monthly_charges = cleaned_data["MonthlyCharges"].mean()
pct_month_to_month = (cleaned_data["Contract"] == "Month-to-month").mean()
pct_fibre_optic = (cleaned_data["InternetService"] == "Fiber optic").mean()
pct_senior_citizen = (cleaned_data["SeniorCitizen"] == "Yes").mean()
avg_tenure_churned = cleaned_data.loc[cleaned_data["Churn"] == 1, "tenure"].mean()
avg_tenure_retained = cleaned_data.loc[cleaned_data["Churn"] == 0, "tenure"].mean()
avg_charges_churned = cleaned_data.loc[cleaned_data["Churn"] == 1, "MonthlyCharges"].mean()
avg_charges_retained = cleaned_data.loc[cleaned_data["Churn"] == 0, "MonthlyCharges"].mean()

eda_summary_rows = [
    {"metric": "total_customers", "value": total_customers},
    {"metric": "churn_rate", "value": round(churn_rate, 4)},
    {"metric": "avg_tenure", "value": round(avg_tenure, 2)},
    {"metric": "avg_monthly_charges", "value": round(avg_monthly_charges, 2)},
    {"metric": "pct_month_to_month", "value": round(pct_month_to_month, 4)},
    {"metric": "pct_fibre_optic", "value": round(pct_fibre_optic, 4)},
    {"metric": "pct_senior_citizen", "value": round(pct_senior_citizen, 4)},
    {"metric": "avg_tenure_churned", "value": round(avg_tenure_churned, 2)},
    {"metric": "avg_tenure_retained", "value": round(avg_tenure_retained, 2)},
    {"metric": "avg_charges_churned", "value": round(avg_charges_churned, 2)},
    {"metric": "avg_charges_retained", "value": round(avg_charges_retained, 2)},
]

eda_summary = pd.DataFrame(eda_summary_rows)
eda_summary_path = project_path("data", "powerbi", "eda_summary.csv")
save_dataframe(eda_summary, eda_summary_path)
file_summary.append(("eda_summary.csv", eda_summary.shape[0], eda_summary.shape[1], os.path.getsize(eda_summary_path)))

✅ Saved: /Users/prasadlola/Documents/customer_churn_project/data/powerbi/eda_summary.csv (275.0B) — 11 rows, 2 columns


## Summary — all 7 files

In [9]:
from utils import format_size

summary_table = pd.DataFrame(file_summary, columns=["File", "Rows", "Columns", "Size_bytes"])
summary_table["Size"] = summary_table["Size_bytes"].apply(format_size)
summary_table = summary_table[["File", "Rows", "Columns", "Size"]]

print("All Power BI files saved successfully:\n")
print(summary_table.to_string(index=False))

All Power BI files saved successfully:

                       File  Rows  Columns    Size
   customer_risk_scores.csv  7032       13 667.0KB
       model_comparison.csv     4        6  478.0B
       churn_by_segment.csv    18        5  810.0B
        ab_test_results.csv     2        6  182.0B
     feature_importance.csv    15        3  583.0B
monthly_revenue_at_risk.csv     3        5  222.0B
            eda_summary.csv    11        2  275.0B
